# Pandas - zadania 41-50 (opcjonalne): rozwiązania

Zakres: czyszczenie i przygotowanie danych o cenach używanych aut.

Część zadań ma kilka równoważnych rozwiązań - poniżej pokazano najczęstsze.

### Import bibliotek i dane

Zadania opcjonalne – rezerwa na wypadek wolnego czasu. Pracujemy na danych o cenach używanych aut w Indiach (plik `data.csv`, 6019 ogłoszeń).

| Kolumna | Znaczenie |
|---|---|
| `Name` | marka i model auta |
| `Location` | miasto |
| `Year` | rok produkcji |
| `Kilometers_Driven` | przebieg w km |
| `Fuel_Type`, `Transmission`, `Owner_Type` | paliwo, skrzynia biegów, który to właściciel |
| `Mileage`, `Engine`, `Power`, `Seats` | zużycie paliwa, pojemność silnika, moc, liczba miejsc |
| `New_Price` | cena nowego auta (tekst: `Lakh` lub `Cr`) |
| `Price` | cena używanego auta w lakhach rupii (1 lakh = 100 000 INR) |

Ćwiczenia są kolejnymi krokami jednego procesu – rozwiązuj je po kolei, bo tabela `auta` przechodzi z ćwiczenia do ćwiczenia.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

Bad value in file PosixPath('/Users/tomasz/.matplotlib/stylelib/company.mplstyle'), line 3 ("axes.prop_cycle: cycler('color', ['#123D98', '#C21E3C', '#E2A713', '#00AEEF', '#7F3F98'])"): Key axes.prop_cycle: "cycler('color', ['" is not a valid cycler construction: Could not parse "cycler('color', ['": unterminated string literal (detected at line 1) (<unknown>, line 1)
Bad value in file PosixPath('/Users/tomasz/.matplotlib/stylelib/company.mplstyle'), line 6 ('grid.color: #e0e0e0'): Key grid.color: '' does not look like a color arg


### Ćwiczenie 41
Wczytaj plik `data.csv` do tabeli `auta`, usuń zbędną kolumnę `Unnamed: 0` i wyświetl trzy pierwsze wiersze. Następnie wypisz wymiary tabeli oraz liczbę zduplikowanych wierszy.

__Oczekiwany wynik (dla drugiej części):__

```
(6019, 13)
0
```

__Wskazówka:__ Użyj `pd.read_csv()`, `drop(columns=...)`, `head()`, `shape` i `duplicated().sum()`. Kolejne ćwiczenia modyfikują tabelę `auta`, więc rozwiązuj je po kolei.

In [2]:
auta = pd.read_csv('data.csv').drop(columns='Unnamed: 0')
auta.head(3)

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,Maruti Wagon R LXI CNG,Mumbai,2010,72000,CNG,Manual,First,26.6 km/kg,998 CC,58.16 bhp,5.0,NaN,1.75
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,18.2 kmpl,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50


In [3]:
print(auta.shape)
print(auta.duplicated().sum())

(6019, 13)
0


### Ćwiczenie 42
Oblicz, jaki procent wartości brakuje w każdej kolumnie. Wyświetl tylko kolumny z jakimkolwiek brakiem, posortowane malejąco, zaokrąglone do 2 miejsc po przecinku.

__Oczekiwany wynik:__

```
New_Price    86.31
Seats         0.70
Engine        0.60
Power         0.60
Mileage       0.03
dtype: float64
```

__Wskazówka:__ `auta.isna().mean()` daje udział braków w każdej kolumnie. Pomnóż go przez 100, przefiltruj maską logiczną i posortuj.

In [4]:
braki = auta.isna().mean() * 100
braki[braki > 0].sort_values(ascending=False).round(2)

New_Price    86.31
Seats         0.70
Engine        0.60
Power         0.60
Mileage       0.03
dtype: float64

### Ćwiczenie 43
Kolumna `Mileage` zawiera zużycie paliwa zapisane tekstem, np. `'19.67 kmpl'` albo `'26.6 km/kg'`.

a) Utwórz kolumnę `Mileage_wartość` (liczba) i `Mileage_jednostka` (tekst po spacji). Policz, ile ogłoszeń ma każdą jednostkę.

b) Wartość `0.0` oznacza w rzeczywistości brak danych. Zamień zera w `Mileage_wartość` na `NaN` i policz braki w tej kolumnie.

__Oczekiwany wynik:__

```
Mileage_jednostka
kmpl     5951
km/kg      66
Name: count, dtype: int64
```

```
70
```

__Wskazówka:__ Użyj `str.split()` z `.str[0]` i `.str[1]`, `pd.to_numeric(..., errors='coerce')` oraz `replace(0, np.nan)`.

In [5]:
auta['Mileage_wartość'] = pd.to_numeric(auta['Mileage'].str.split().str[0], errors='coerce')
auta['Mileage_jednostka'] = auta['Mileage'].str.split().str[1]
auta['Mileage_jednostka'].value_counts()

Mileage_jednostka
kmpl     5951
km/kg      66
Name: count, dtype: int64

In [6]:
auta['Mileage_wartość'] = auta['Mileage_wartość'].replace(0, np.nan)
print(auta['Mileage_wartość'].isna().sum())

70


### Ćwiczenie 44
Utwórz liczbowe kolumny `Engine_cc` (z kolumny `Engine`, np. `'998 CC'`) oraz `Power_bhp` (z kolumny `Power`, np. `'58.16 bhp'`, czasem `'null bhp'`). Nieczytelne wartości mają stać się brakami. Wyświetl statystyki opisowe obu nowych kolumn, zaokrąglone do 1 miejsca.

__Oczekiwany wynik:__

```
       Engine_cc  Power_bhp
count     5983.0     5876.0
mean      1621.3      113.3
std        601.4       53.9
min         72.0       34.2
25%       1198.0       75.0
50%       1493.0       97.7
75%       1984.0      138.1
max       5998.0      560.0
```

__Wskazówka:__ Usuń jednostkę przez `str.replace()`, a tekst zamień na liczby przez `pd.to_numeric(..., errors='coerce')`.

In [7]:
auta['Engine_cc'] = pd.to_numeric(auta['Engine'].str.replace(' CC', ''), errors='coerce')
auta['Power_bhp'] = pd.to_numeric(auta['Power'].str.replace(' bhp', ''), errors='coerce')
auta[['Engine_cc', 'Power_bhp']].describe().round(1)

,Engine_cc,Power_bhp
count,5983.0,5876.0
mean,1621.3,113.3
std,601.4,53.9
min,72.0,34.2
25%,1198.0,75.0
50%,1493.0,97.7
75%,1984.0,138.1
max,5998.0,560.0


### Ćwiczenie 45
Kolumna `New_Price` zawiera cenę nowego auta jako tekst: `'8.61 Lakh'` lub `'1.2 Cr'` (1 Cr = 100 Lakh). Utwórz kolumnę `New_Price_lakh` z ceną wyrażoną w lakhach (liczba). Wyświetl jej statystyki opisowe zaokrąglone do 1 miejsca.

__Oczekiwany wynik:__

```
count    824.0
mean      23.1
std       26.2
min        3.9
25%        8.0
50%       11.7
75%       26.6
max      230.0
Name: New_Price_lakh, dtype: float64
```

__Wskazówka:__ Rozdziel tekst przez `str.split()`. Liczbę (`.str[0]`) zamień przez `pd.to_numeric()`, jednostkę (`.str[1]`) zamień na mnożnik metodą `map({'Lakh': 1, 'Cr': 100})`, a następnie pomnóż.

In [8]:
części = auta['New_Price'].str.split()
wartość = pd.to_numeric(części.str[0])
mnożnik = części.str[1].map({'Lakh': 1, 'Cr': 100})
auta['New_Price_lakh'] = wartość * mnożnik
auta['New_Price_lakh'].describe().round(1)

count    824.0
mean      23.1
std       26.2
min        3.9
25%        8.0
50%       11.7
75%       26.6
max      230.0
Name: New_Price_lakh, dtype: float64

### Ćwiczenie 46
a) Utwórz kolumnę `Marka` z pierwszego słowa kolumny `Name`. Wyświetl pięć najczęstszych marek.

b) Ujednolić nazwy: `'ISUZU'` zamień na `'Isuzu'`, a `'Land'` (skrócona nazwa marki Land Rover) na `'Land Rover'`. Wypisz liczbę różnych marek.

__Oczekiwany wynik:__

```
Marka
Maruti           1211
Hyundai          1107
Honda             608
Toyota            411
Mercedes-Benz     318
Name: count, dtype: int64
```

```
30
```

__Wskazówka:__ Użyj `str.split().str[0]`, `value_counts()`, `replace()` ze słownikiem i `nunique()`.

In [9]:
auta['Marka'] = auta['Name'].str.split().str[0]
auta['Marka'].value_counts().head(5)

Marka
Maruti           1211
Hyundai          1107
Honda             608
Toyota            411
Mercedes-Benz     318
Name: count, dtype: int64

In [10]:
auta['Marka'] = auta['Marka'].replace({'ISUZU': 'Isuzu', 'Land': 'Land Rover'})
print(auta['Marka'].nunique())

30


### Ćwiczenie 47
W kolumnach `Engine_cc`, `Power_bhp` i `Seats` są braki.

a) Uzupełnij braki w `Engine_cc` i `Power_bhp` **medianą dla marki** danego auta.

b) Braki w `Seats` uzupełnij medianą z całej kolumny.

c) Wypisz liczbę braków w tych trzech kolumnach.

__Oczekiwany wynik:__

```
Engine_cc    0
Power_bhp    1
Seats        0
dtype: int64
```

__Wskazówka:__ Mediany dla marki policzysz przez `groupby('Marka')['Engine_cc'].transform('median')`. Użyj ich w `fillna()` i przypisz wynik z powrotem do kolumny. Marki bez żadnej wartości nadal będą miały brak.

In [ ]:
for kolumna in ['Engine_cc', 'Power_bhp']:
    auta[kolumna] = auta[kolumna].fillna(auta.groupby('Marka')[kolumna].transform('median'))
auta['Seats'] = auta['Seats'].fillna(auta['Seats'].median())
auta[['Engine_cc', 'Power_bhp', 'Seats']].isna().sum()

### Ćwiczenie 48
Dane pochodzą z 2019 roku.

a) Dodaj kolumnę `wiek` (2019 minus `Year`) oraz `km_na_rok` – średni roczny przebieg (`Kilometers_Driven` podzielony przez wiek; wiek 0 traktuj jako 1). Wyświetl statystyki opisowe `km_na_rok` zaokrąglone do całości.

b) Wyświetl trzy auta z największym przebiegiem (kolumny `Name`, `Year`, `Kilometers_Driven`). Pierwszy wynik jest podejrzany (6,5 mln km).

c) Usuń ogłoszenia z przebiegiem co najmniej 1 000 000 km i wypisz nowe wymiary tabeli.

__Oczekiwany wynik:__

```
count       6019.0
mean       12755.0
std        42673.0
min          105.0
25%         6934.0
50%        10125.0
75%        14798.0
max      3250000.0
Name: km_na_rok, dtype: float64
```

```
                                        Name  Year  Kilometers_Driven
2328               BMW X5 xDrive 30d M Sport  2017            6500000
340   Skoda Octavia Ambition Plus 2.0 TDI AT  2013             775000
1860        Volkswagen Vento Diesel Highline  2013             720000
```

```
(6018, 21)
```

__Wskazówka:__ Użyj `clip(lower=1)`, `describe()`, `nlargest()` oraz filtrowania maską logiczną.

In [ ]:
auta['wiek'] = 2019 - auta['Year']
auta['km_na_rok'] = auta['Kilometers_Driven'] / auta['wiek'].clip(lower=1)
auta['km_na_rok'].describe().round(0)

In [ ]:
auta.nlargest(3, 'Kilometers_Driven')[['Name', 'Year', 'Kilometers_Driven']]

In [ ]:
auta = auta[auta['Kilometers_Driven'] < 1_000_000]
print(auta.shape)

### Ćwiczenie 49
a) Zamień `Owner_Type` na liczbę właścicieli (`First` = 1, `Second` = 2, `Third` = 3, `Fourth & Above` = 4) w kolumnie `Właściciele`. Policz auta w każdej grupie.

b) Utwórz kolumnę `Automat` równą 1 dla skrzyni `'Automatic'` i 0 dla pozostałych. Wypisz udział aut z automatyczną skrzynią (zaokrąglony do 3 miejsc).

__Oczekiwany wynik:__

```
Właściciele
1    4928
2     968
3     113
4       9
Name: count, dtype: int64
```

```
0.286
```

__Wskazówka:__ Użyj `map()` ze słownikiem, `value_counts()`, porównania `==` oraz `astype(int)`. Średnia z kolumny zer i jedynek to udział jedynek.

In [ ]:
auta['Właściciele'] = auta['Owner_Type'].map({'First': 1, 'Second': 2, 'Third': 3, 'Fourth & Above': 4})
auta['Właściciele'].value_counts()

In [ ]:
auta['Automat'] = (auta['Transmission'] == 'Automatic').astype(int)
print(auta['Automat'].mean().round(3))

### Ćwiczenie 50
a) Zbuduj „raport jakości danych”: tabelę `raport`, w której wierszami są kolumny tabeli `auta`, a kolumny to `typ` (typ danych jako tekst), `unikalne` (liczba różnych wartości) i `braki` (liczba braków). Wyświetl ją.

b) Zapisz oczyszczoną tabelę do pliku `auta_czyste.csv` (bez indeksu), wczytaj ją z powrotem i sprawdź, czy wymiary się zgadzają. Na koniec usuń plik.

__Oczekiwany wynik (dla części b):__

```
True
```

__Wskazówka:__ Typy to `auta.dtypes.astype(str)`, liczby unikalnych wartości – `auta.nunique()`. Plik usuniesz funkcją `os.remove()`.

In [ ]:
raport = pd.DataFrame({'typ': auta.dtypes.astype(str),
                       'unikalne': auta.nunique(),
                       'braki': auta.isna().sum()})
raport

In [ ]:
auta.to_csv('auta_czyste.csv', index=False)
wczytane = pd.read_csv('auta_czyste.csv')
print(wczytane.shape == auta.shape)
os.remove('auta_czyste.csv')